# Week 43, Day 1: Stacks

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w043d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
stack = []
for token in "a b ( c d ) e ) f".split():
    if token == ")":
        if stack:
            stack.pop()
    else:
        stack.append(token)
print(stack, stack[-1] if stack else None, len(stack))

In [ ]:
class ArrayStack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        if not self._items:
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def peek(self):
        if not self._items:
            raise IndexError("peek at empty stack")
        return self._items[-1]

    def __len__(self):
        return len(self._items)

    def __repr__(self):
        return f"ArrayStack({self._items!r})"

s = ArrayStack()
for x in [1, 2, 3]:
    s.push(x)
print(s, s.pop(), s.peek(), len(s))

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

class LinkedStack:
    def __init__(self):
        self._top = None
        self._size = 0

    def push(self, item):
        self._top = Node(item, self._top)
        self._size += 1

    def pop(self):
        if self._top is None:
            raise IndexError("pop from empty stack")
        item, self._top = self._top.value, self._top.next
        self._size -= 1
        return item

    def peek(self):
        if self._top is None:
            raise IndexError("peek at empty stack")
        return self._top.value

    def __len__(self):
        return self._size

def drain(stack):
    out = []
    while len(stack):
        out.append(stack.pop())
    return out

s = LinkedStack()
for x in "abc":
    s.push(x)
print(s.peek(), len(s), drain(s), len(s))

In [ ]:
import traceback

def depth(n):
    if n == 0:
        return len(traceback.extract_stack())       # how many frames are on the call stack now
    return depth(n - 1)

base = depth(0)
print(depth(10) - base, depth(100) - base)        # one extra frame per extra call

In [ ]:
grid = [list(row) for row in [
    "..#.....",
    ".##..#..",
    "....##..",
    "#...#...",
]]

def fill_recursive(grid, r, c):
    if not (0 <= r < len(grid) and 0 <= c < len(grid[0])) or grid[r][c] != ".":
        return 0
    grid[r][c] = "o"
    return 1 + sum(fill_recursive(grid, r + dr, c + dc) for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)])

def fill_with_stack(grid, r, c):
    filled = 0
    stack = [(r, c)]
    while stack:
        r, c = stack.pop()
        if not (0 <= r < len(grid) and 0 <= c < len(grid[0])) or grid[r][c] != ".":
            continue
        grid[r][c] = "o"
        filled += 1
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            stack.append((r + dr, c + dc))
    return filled

copy = [row[:] for row in grid]
print(fill_recursive(copy, 0, 0), fill_with_stack(grid, 0, 0))
print("\n".join("".join(row) for row in grid))
big = [["."] * 300 for _ in range(300)]
print(fill_with_stack(big, 0, 0))

In [ ]:
class MinStack:
    def __init__(self):
        self._items = []                  # (item, smallest so far)

    def push(self, item):
        smallest = item if not self._items else min(item, self._items[-1][1])
        self._items.append((item, smallest))

    def pop(self):
        return self._items.pop()[0]

    def min(self):
        return self._items[-1][1]

s = MinStack()
for x in [5, 3, 7, 3, 1, 8]:
    s.push(x)
mins = []
while s._items:
    mins.append(s.min())
    s.pop()
print(mins)

> **Key idea.** A stack promises push, pop and peek at the top, all O(1), and nothing else: an abstract data type, which code can rely on whatever the implementation. On a Python list the top is the end; on linked nodes it's the head. The call stack holds a frame per function call, which is why recursion is limited to about 1,000 calls; an explicit stack (push the start, then pop, process, push new work) replaces recursion without that limit. Stacks can carry extra facts, like the minimum so far, in O(1). Nesting, undoing, most-recent-first and depth-first problems are stack-shaped.

### Exercise: A stack on a list

Write `ArrayStack` with `push`, `pop` and `peek` (both raising `IndexError("pop from empty stack")` and `IndexError("peek at empty stack")` when empty), `__len__`, `__bool__`, `__iter__` (from the top down, without changing the stack, and without `yield`) and `__repr__` like `ArrayStack([1, 2, 3])` (bottom first). Keep the items in a Python list with the top at the end.

In [ ]:
class ArrayStack:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stack():\n    \"a stack\"\n    s = ArrayStack()\n    assert not s and len(s) == 0 and repr(s) == \"ArrayStack([])\"\n    for x in [1, 2, 3]:\n        s.push(x)\n    assert repr(s) == \"ArrayStack([1, 2, 3])\" and list(s) == [3, 2, 1] and list(s) == [3, 2, 1] and len(s) == 3 and s\n    assert s.peek() == 3 and s.pop() == 3 and s.pop() == 2 and s.peek() == 1 and len(s) == 1\n    s.pop()\n    for name, message in [(\"pop\", \"pop from empty stack\"), (\"peek\", \"peek at empty stack\")]:\n        try:\n            getattr(s, name)()\n            assert False\n        except IndexError as err:\n            assert str(err) == message\n    assert \"yield\" not in source")

### Exercise: A stack on linked nodes

Write `LinkedStack` with exactly the same behaviour as `ArrayStack` (the checks run the same tests on both), but storing items in `Node`s with the top at the head and a size counter, so every operation is O(1) in the worst case. Its repr is `LinkedStack([1, 2, 3])` (bottom first).

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

class LinkedStack:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stack():\n    \"a linked stack\"\n    s = LinkedStack()\n    assert not s and len(s) == 0 and repr(s) == \"LinkedStack([])\"\n    for x in [1, 2, 3]:\n        s.push(x)\n    assert repr(s) == \"LinkedStack([1, 2, 3])\" and list(s) == [3, 2, 1] and list(s) == [3, 2, 1] and len(s) == 3 and s\n    assert s.peek() == 3 and s.pop() == 3 and s.pop() == 2 and s.peek() == 1 and len(s) == 1\n    s.pop()\n    for name, message in [(\"pop\", \"pop from empty stack\"), (\"peek\", \"peek at empty stack\")]:\n        try:\n            getattr(s, name)()\n            assert False\n        except IndexError as err:\n            assert str(err) == message\n    assert isinstance(s._top, (Node, type(None)))")

### Exercise: Minimum and maximum in O(1)

Write `MinMaxStack` with `push`, `pop` (returns the item), `min()`, `max()` and `__len__`, where `min` and `max` are O(1). Empty `pop`, `min` or `max` raises `IndexError`. Store, with each item, the smallest and largest so far.

In [ ]:
class MinMaxStack:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_minmax():\n    \"against a plain list\"\n    rng = random.Random(1)\n    s, plain = MinMaxStack(), []\n    for _ in range(2000):\n        if plain and rng.random() < 0.4:\n            assert s.pop() == plain.pop()\n        else:\n            x = rng.randint(-50, 50)\n            s.push(x)\n            plain.append(x)\n        if plain:\n            assert s.min() == min(plain) and s.max() == max(plain) and len(s) == len(plain)\n    for name in [\"min\", \"max\", \"pop\"]:\n        try:\n            getattr(MinMaxStack(), name)()\n            assert False, name\n        except IndexError:\n            pass")

### Exercise: Tidying a path

Write `simplify(path)` for an absolute folder path like `"/home//ada/./docs/../music/"`: return the tidy version, `"/home/ada/music"`. Split on `/`; skip empty parts and `.`; `..` goes up one folder (at the root it does nothing); anything else is a folder name. The result starts with `/`, has single slashes and no trailing slash, and is `"/"` for the root. Use a stack of folder names.

In [ ]:
def simplify(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_paths():\n    \"paths\"\n    cases = {\"/home//ada/./docs/../music/\": \"/home/ada/music\", \"/\": \"/\", \"/../\": \"/\", \"/a/b/../../c\": \"/c\", \"/a/./b/./\": \"/a/b\",\n             \"/...\": \"/...\", \"/a/..b/c\": \"/a/..b/c\", \"//x//y//..//\": \"/x\"}\n    for path, want in cases.items():\n        assert simplify(path) == want, (path, simplify(path))")

### Exercise: Islands without recursion

`grid` is a list of strings of `"#"` (land) and `"."` (water). Write `islands(grid)`: the sizes of the islands (groups of land squares joined up, down, left or right), largest first. Use an **explicit stack**, not recursion: the checks use a 400 by 400 island, far too deep for recursion. Don't change `grid` (keep a set of visited squares).

In [ ]:
def islands(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_small():\n    \"small\"\n    assert islands([\"##..\", \"#..#\", \"..##\", \"#...\"]) == [3, 3, 1] and islands([]) == [] and islands([\"....\"]) == [] and islands([\"#\"]) == [1]\n    g = [\"#.#\", \".#.\", \"#.#\"]\n    assert islands(g) == [1, 1, 1, 1, 1] and g == [\"#.#\", \".#.\", \"#.#\"]\ndef test_big():\n    \"one big island\"\n    assert islands([\"#\" * 400] * 400) == [160_000]")

### Exercise: Repeated text

Write `decode(text)` for strings like `"3[ab]2[c]"`, meaning `"abababcc"`: a whole number followed by `[...]` means the bracketed text repeated that many times, and brackets can nest (`"2[a3[b]]"` is `"abbbabbb"`). Letters outside brackets stand for themselves. Use a stack: on `[`, push the text so far and the count, and start fresh; on `]`, pop them and join.

In [ ]:
def decode(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_decode():\n    \"decoding\"\n    cases = {\"3[ab]2[c]\": \"abababcc\", \"2[a3[b]]\": \"abbbabbb\", \"xy\": \"xy\", \"\": \"\", \"10[z]\": \"z\" * 10, \"a2[b]c\": \"abbc\", \"2[3[2[x]]]\": \"x\" * 12, \"1[]q\": \"q\"}\n    for text, want in cases.items():\n        assert decode(text) == want, (text, decode(text))")

### Bug hunt: A silent empty pop

This stack's `pop` returns `None` when it's empty instead of raising, so the mistake in the calling code (one pop too many) goes unnoticed and `None` ends up in the results: this prints `[3, 2, 1, None]`. Make `pop` raise `IndexError` (Week 37: fail fast), so the program prints `[3, 2, 1]` and then `IndexError: pop from empty stack`.

In [ ]:
class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        if self._items:
            return self._items.pop()

s = Stack()
for x in [1, 2, 3]:
    s.push(x)
out = []
try:
    for _ in range(4):
        out.append(s.pop())
except IndexError as err:
    print(out)
    print("IndexError:", err)
else:
    print(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fail fast\"\n    expect_output(\"[3, 2, 1]\\nIndexError: pop from empty stack\")")

### Bug hunt: A minimum that forgets

This min-stack keeps a single `smallest` number instead of one per item, so after popping the minimum it still reports it: this prints `[1, 1, 1, 1]`. Store the minimum so far with each item, so it prints `[1, 3, 3, 5]`.

In [ ]:
class MinStack:
    def __init__(self):
        self._items = []
        self.smallest = None

    def push(self, item):
        self._items.append(item)
        if self.smallest is None or item < self.smallest:
            self.smallest = item

    def pop(self):
        return self._items.pop()

    def min(self):
        return self.smallest

s = MinStack()
for x in [5, 3, 7, 1]:
    s.push(x)
mins = []
for _ in range(4):
    mins.append(s.min())
    s.pop()
print(mins)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a minimum per item\"\n    expect_output(\"[1, 3, 3, 5]\")")

### Bug hunt: Above the root

`..` at the root should do nothing, but this pops from an empty stack and crashes with `IndexError`. Fix it so this prints `/b`.

In [ ]:
def simplify(path):
    stack = []
    for part in path.split("/"):
        if part in ("", "."):
            continue
        if part == "..":
            stack.pop()
        else:
            stack.append(part)
    return "/" + "/".join(stack)

print(simplify("/a/../../b"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"nothing above the root\"\n    expect_output(\"/b\")")

### Puzzle: An explicit stack

Arrange the lines into a function that counts the squares reachable from `start` in a grid, with an explicit stack. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
s = []
for x in [4, 1, 3]:
    s.append(x)
a = s.pop()
s.append(9)
b = s[-1]
c = len(s)

p = ___   # a
q = ___   # b
r = ___   # c
t = ___   # which end of a Python list should be a stack's top: "start" or "end"
u = ___   # roughly how many nested calls CPython allows before RecursionError

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 3 and q == 9 and r == 3, \"p, q and r\"\n    assert t == \"end\" and u == 1000, \"t and u\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: asteroid collisions

Asteroids fly along a line. Each is a non-zero whole number: its size is the absolute value, and its sign is its direction (positive moves right, negative moves left). They all move at the same speed. When a right-mover meets a left-mover, the smaller explodes; if they're the same size, both do. Two asteroids moving the same way never meet, and a left-mover that's to the left of everything moving right just flies off. Write `survivors(asteroids)`: the asteroids left after every collision, in order. Use a stack in one pass (O(n)); the checks use 100,000 asteroids.

In [ ]:
def survivors(asteroids):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef slow(asteroids):\n    a = list(asteroids)\n    changed = True\n    while changed:\n        changed = False\n        for i in range(len(a) - 1):\n            if a[i] > 0 > a[i + 1]:\n                if a[i] > -a[i + 1]:\n                    del a[i + 1]\n                elif a[i] < -a[i + 1]:\n                    del a[i]\n                else:\n                    del a[i:i + 2]\n                changed = True\n                break\n    return a\ndef test_small():\n    \"small\"\n    assert survivors([5, 10, -5]) == [5, 10] and survivors([8, -8]) == [] and survivors([10, 2, -5]) == [10] and survivors([-2, -1, 1, 2]) == [-2, -1, 1, 2]\n    rng = random.Random(3)\n    for _ in range(300):\n        a = [rng.choice([-1, 1]) * rng.randint(1, 6) for _ in range(rng.randint(0, 10))]\n        assert survivors(a) == slow(a), a\ndef test_big():\n    \"100,000 asteroids\"\n    rng = random.Random(4)\n    a = [rng.choice([-1, 1]) * rng.randint(1, 1000) for _ in range(100_000)]\n    got = survivors(a)\n    assert all(not (x > 0 > y) for x, y in zip(got, got[1:]))")

## Warm-up

Every `)` removed whatever was most recently added, whether or not it was a `(`. That's all a stack does: it always hands back the **most recent** item. The art is in deciding what to put on it.

## A stack is a promise

A **stack** is defined by what it does, not how:

| Operation | Does | Cost (promised) |
|---|---|---|
| `push(x)` | put `x` on top | O(1) |
| `pop()` | remove and return the top item (error if empty) | O(1) |
| `peek()` | return the top item without removing it (error if empty) | O(1) |
| `len(s)`, `bool(s)` | how many items; is it empty? | O(1) |

This kind of definition, a set of operations and their behaviour with no mention of storage, is called an **abstract data type** (ADT). It's Week 32's Protocol idea, applied to data structures: code that uses a stack depends only on the promise, so the implementation underneath can change. The idea is usually credited to Barbara Liskov and Stephen Zilles's 1974 paper "Programming with Abstract Data Types" (the same Liskov as Week 34's substitution principle).

Everything else is deliberately missing: no indexing, no looking at the middle, no inserting below the top. Those restrictions are the point: a stack can't be misused as something else, and every operation stays O(1).

## Two implementations

On a Python list, the top is the **end**, because `append` and `pop()` are O(1) there (Week 41) while the front is O(n):

On linked nodes (Week 42), the top is the **head**, because adding and removing at the head is O(1):

`drain` doesn't know or care which kind of stack it gets: that's the ADT doing its job. Which is better? The list version is usually faster in Python (one object, items side by side, amortised O(1) push); the linked version has a true O(1) worst case for every push, and never wastes spare capacity. For almost every Python program, use a list.

## The call stack

Every running Python program has a stack you never wrote: the **call stack**. Each function call pushes a **frame** holding its local variables and where to return to; each `return` pops it. That's why a function can call itself (Week 15): each call gets its own frame. And it's why deep recursion fails: CPython limits the call stack to about 1,000 frames, then raises `RecursionError`.

Any recursive algorithm can be rewritten with an **explicit stack** of your own, which can grow as big as memory allows. The pattern: push the starting work; loop while the stack isn't empty; pop a piece of work, do it, and push any new work it creates. Here's Week 16's flood fill, done both ways:

The recursive version would hit the recursion limit on the 300 by 300 grid; the stack version fills all 90,000 squares. (The order squares are filled in differs, but the result is the same.)

## Stacks that remember more

A stack can carry extra information with each item, as long as it can be worked out from the item and the one below it. The classic is a stack that can report its **minimum** in O(1): push, with each item, the minimum of everything up to and including it. Popping an item uncovers the minimum of what was below it, already worked out:

Searching the stack for its minimum would be O(n) per call; the stored minimums make it O(1), at the cost of one extra value per item.

## Stack-shaped problems

Reach for a stack when the problem has one of these shapes:

| Shape | Examples |
|---|---|
| **nesting**: things open and must close in reverse order | brackets, HTML tags, nested lists in text, folder paths with `..` |
| **undoing**: reverse the most recent action first | undo, back buttons, backtracking in a search |
| **most recent first**: the newest item matters until it's dealt with | parsing expressions, collisions, "the nearest bigger number" (Day 3) |
| **depth first**: finish one branch before the next | searching mazes, trees and folders without recursion |

## Practice

## Quiz

## Daily challenge

## Before you go

Find a recursive function you've written (a tree walk, a flood fill, a search) and rewrite it with an explicit stack. Tomorrow: queues and deques, the first-in, first-out siblings of stacks, and why `collections.deque` exists.